# Limpieza de datos

Se trabajará con datos bajados de Kaggle (https://www.kaggle.com/datasets/yagunnersya/fifa-21-messy-raw-dataset-for-cleaning-exploring?resource=download) de datos del videojuego FIFA 21, que contiene datos sucios.

Para empezar, analizaremos la información de la tabla de datos

In [37]:
import pandas as pd
file_path = 'fifa21 raw data v2.csv'
df = pd.read_csv(file_path, low_memory=False)
# Mostrar SIEMPRE el detalle completo de columnas en info()
pd.set_option('display.max_info_columns', 1000)

# Si quieres ver exactamente las 76 columnas originales, ejecuta esta celda antes de crear columnas nuevas
# (o usa df.iloc[:, :76].info(...)).
df.info(verbose=True, show_counts=True)
# Joined y Loan Date end tendria que ser fecha
# Contract
# W/F, SM, IR que son?

<class 'pandas.DataFrame'>
RangeIndex: 18979 entries, 0 to 18978
Data columns (total 77 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   ID                18979 non-null  int64
 1   Name              18979 non-null  str  
 2   LongName          18979 non-null  str  
 3   photoUrl          18979 non-null  str  
 4   playerUrl         18979 non-null  str  
 5   Nationality       18979 non-null  str  
 6   Age               18979 non-null  int64
 7   ↓OVA              18979 non-null  int64
 8   POT               18979 non-null  int64
 9   Club              18979 non-null  str  
 10  Contract          18979 non-null  str  
 11  Positions         18979 non-null  str  
 12  Height            18979 non-null  str  
 13  Weight            18979 non-null  str  
 14  Preferred Foot    18979 non-null  str  
 15  BOV               18979 non-null  int64
 16  Best Position     18979 non-null  str  
 17  Joined            18979 non-null  str  
 1

## Posibles problemas:
1. Value, Wage, Release Clause, Height, Weight y Hits tendrían que ser numéricas y están como "str"
2. Joined y Loan Date End tendrían que ser fechas y también están como "str".
3. W/F, SM, e IR aparecen en Kaggle con una estrella, veremos cómo aparecen en el df
4. Contract podría tener potenciales problemas también ya que en Kaggle están en un formato extraño

Como siguiente paso, se imprimirán las primeras 5 filas de todas las columnas de las que sospechamos para identificar el problema

In [38]:
df[['Value', 'Wage', 'Release Clause', 'Height', 'Weight', 'Hits', 'Contract', 'Joined', 'Loan Date End', 'W/F', 'SM', 'IR']].head()

,Value,Wage,Release Clause,Height,Weight,Hits,Contract,Joined,Loan Date End,W/F,SM,IR
0,€103.5M,€560K,€138.4M,170cm,72kg,771,2004 ~ 2021,"Jul 1, 2004",NaN,4 ★,4★,5 ★
1,€63M,€220K,€75.9M,187cm,83kg,562,2018 ~ 2022,"Jul 10, 2018",NaN,4 ★,5★,5 ★
2,€120M,€125K,€159.4M,188cm,87kg,150,2014 ~ 2023,"Jul 16, 2014",NaN,3 ★,1★,3 ★
3,€129M,€370K,€161M,181cm,70kg,207,2015 ~ 2023,"Aug 30, 2015",NaN,5 ★,4★,4 ★
4,€132M,€270K,€166.5M,175cm,68kg,595,2017 ~ 2022,"Aug 3, 2017",NaN,5 ★,5★,5 ★


* Se observa que las primeras columnas tienen la unidad, por lo que habria que sacarla y convertirlas a numéricas. 
* Los contratos estan separados por un '~', habría que dividirlos en dos columnas: inicio de contrato y fin de contrato
* Hay que pasar Joined a formato fecha
* Loan Date End tiene muchos NAs porque para tener un valor hay que estar a préstamo, y este no es el caso de la mayoría de los jugadores. Se dejará como está ya que no aporta mucha información
* En las últimas tres columnas se puede notar la estrella. Se elminará y se dará por entendido que son valoraciones del 1 al 5

### 1) Value, Wage, Release Clause, Height, Weight y Hits

In [39]:

import numpy as np
import re

# Empezaremos con la limpieza de las columnas ['Value', 'Wage', 'Release Clause', 'Height', 'Weight', 'Hits']
def parse_money(value):
    """Convierte valores como €1.5M, €500K, €0 o '-' a número (EUR)."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)

    s = str(value).strip().replace('€', '').replace(',', '')
    if s in {'', '-', 'nan'}:
        return np.nan

    mult = 1
    if s.endswith('M'):
        mult = 1_000_000
        s = s[:-1]
    elif s.endswith('K'):
        mult = 1_000
        s = s[:-1]

    try:
        return float(s) * mult
    except ValueError:
        return np.nan


def parse_height_cm(value):
    """Convierte altura a cm desde formatos como 5'11" o 180cm."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)

    s = str(value).strip().lower()
    if s in {'', '-', 'nan'}:
        return np.nan

    # Formato pies+pulgadas: 5'11
    m = re.match(r"^(\d+)'(\d+)$", s)
    if m:
        feet = int(m.group(1))
        inches = int(m.group(2))
        return round((feet * 30.48) + (inches * 2.54), 2)

    # Formato cm: 180cm
    s = s.replace('cm', '').strip()
    try:
        return float(s)
    except ValueError:
        return np.nan


def parse_weight_kg(value):
    """Convierte peso a kg desde formatos como 165lbs o 75kg."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)

    s = str(value).strip().lower()
    if s in {'', '-', 'nan'}:
        return np.nan

    if 'lbs' in s:
        n = s.replace('lbs', '').strip()
        try:
            return round(float(n) * 0.453592, 2)
        except ValueError:
            return np.nan

    s = s.replace('kg', '').strip()
    try:
        return float(s)
    except ValueError:
        return np.nan


def parse_hits(value):
    """Convierte hits desde formatos como 1.2K, 900, '-' a número."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)

    s = str(value).strip().replace(',', '').upper()
    if s in {'', '-', 'NAN'}:
        return np.nan

    mult = 1
    if s.endswith('K'):
        mult = 1_000
        s = s[:-1]

    try:
        return float(s) * mult
    except ValueError:
        return np.nan


# Crea columnas numéricas limpias
if 'Value' in df.columns:
    df['Value_EUR'] = df['Value'].apply(parse_money)
if 'Wage' in df.columns:
    df['Wage_EUR'] = df['Wage'].apply(parse_money)
if 'Release Clause' in df.columns:
    df['Release Clause_EUR'] = df['Release Clause'].apply(parse_money)
if 'Height' in df.columns:
    df['Height_cm'] = df['Height'].apply(parse_height_cm)
if 'Weight' in df.columns:
    df['Weight_kg'] = df['Weight'].apply(parse_weight_kg)
if 'Hits' in df.columns:
    df['Hits'] = df['Hits'].apply(parse_hits)

# DataFrame listo para trabajar (mantiene originales y agrega numéricas)
df_limpio = df.drop(columns=['Value', 'Wage', 'Release Clause', 'Height', 'Weight'])

print("\nInformación de las columnas del DataFrame limpio:")
df_limpio[['Value_EUR', 'Wage_EUR', 'Release Clause_EUR', 'Height_cm', 'Weight_kg', 'Hits']].info()


Información de las columnas del DataFrame limpio:
<class 'pandas.DataFrame'>
RangeIndex: 18979 entries, 0 to 18978
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Value_EUR           18979 non-null  float64
 1   Wage_EUR            18979 non-null  float64
 2   Release Clause_EUR  18979 non-null  float64
 3   Height_cm           18939 non-null  float64
 4   Weight_kg           18979 non-null  float64
 5   Hits                16384 non-null  float64
dtypes: float64(6)
memory usage: 889.8 KB


In [40]:
print("Columnas limpias creadas:")
df_limpio[['Value_EUR', 'Wage_EUR', 'Release Clause_EUR', 'Height_cm', 'Weight_kg', 'Hits']].head()

Columnas limpias creadas:


,Value_EUR,Wage_EUR,Release Clause_EUR,Height_cm,Weight_kg,Hits
0,103500000.0,560000.0,138400000.0,170.0,72.0,771.0
1,63000000.0,220000.0,75900000.0,187.0,83.0,562.0
2,120000000.0,125000.0,159400000.0,188.0,87.0,150.0
3,129000000.0,370000.0,161000000.0,181.0,70.0,207.0
4,132000000.0,270000.0,166500000.0,175.0,68.0,595.0


### 2) Contracts

In [41]:
# 2) Separar columna Contract en inicio y fin

# Divide por '~': izquierda = inicio, derecha = fin
contract_parts = df_limpio['Contract'].astype(str).str.split('~', n=1, expand=True)

df_limpio['contract_begins'] = contract_parts[0].str.strip()
df_limpio['contract_ends'] = contract_parts[1].str.strip() if contract_parts.shape[1] > 1 else pd.NA

# Extrae solo anios (YYYY) y los guarda como enteros anulables
for col in ['contract_begins', 'contract_ends']:
    df_limpio[col] = pd.to_numeric(
        df_limpio[col].str.extract(r'(\d{4})', expand=False),
        errors='coerce'
    ).astype('Int64')

df_limpio= df_limpio.drop(columns=['Contract'])

print("Vista de Contract separado:")
print(df_limpio[['contract_begins', 'contract_ends']].head(10))

print("Tipo de datos:")
print(df_limpio[['contract_begins', 'contract_ends']].dtypes)

Vista de Contract separado:
   contract_begins  contract_ends
0             2004           2021
1             2018           2022
2             2014           2023
3             2015           2023
4             2017           2022
5             2014           2023
6             2017           2023
7             2018           2024
8             2018           2022
9             2014           2022
Tipo de datos:
contract_begins    Int64
contract_ends      Int64
dtype: object


### 3) Joined

In [42]:
# 3) Convertir Joined a formato fecha
if 'Joined' not in df_limpio.columns:
    raise ValueError("No existe la columna 'Joined' en df_limpio.")

# Convierte a datetime; valores no parseables quedan como NaT
df_limpio['Joined'] = pd.to_datetime(df_limpio['Joined'], errors='coerce')

print("Tipo de dato de Joined:")
print(df_limpio['Joined'].dtype)

print("\nPrimeras filas de Joined:")
print(df_limpio['Joined'].head(10))

print("\nCantidad de fechas no convertidas (NaT):")
print(df_limpio['Joined'].isna().sum())

Tipo de dato de Joined:
datetime64[us]

Primeras filas de Joined:
0   2004-07-01
1   2018-07-10
2   2014-07-16
3   2015-08-30
4   2017-08-03
5   2014-07-01
6   2017-07-01
7   2018-07-19
8   2018-07-01
9   2014-07-01
Name: Joined, dtype: datetime64[us]

Cantidad de fechas no convertidas (NaT):
0


### 4) 'W/F', 'SM' e 'IR'

In [43]:
# 4) Eliminar '*' de W/F, SM e IR y convertir a numerico
cols_estrellas = ['W/F', 'SM', 'IR']

for col in cols_estrellas:
    if col in df_limpio.columns:
        df_limpio[col] = (
            df_limpio[col]
            .astype(str)
            .str.replace('★', '', regex=False)
            .str.strip()
        )
        df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce').astype('Int64')

print("Vista previa sin estrella:")
print(df_limpio[cols_estrellas].head(10))

print("\nTipos de dato:")
print(df_limpio[cols_estrellas].dtypes)

Vista previa sin estrella:
   W/F  SM  IR
0    4   4   5
1    4   5   5
2    3   1   3
3    5   4   4
4    5   5   5
5    4   4   4
6    3   4   3
7    3   1   3
8    4   5   3
9    4   1   3

Tipos de dato:
W/F    Int64
SM     Int64
IR     Int64
dtype: object


## CONCLUSIÓN

Se modificaron todas las columnas en donde se encontró un error. Mostraremos la dimensión del DataFrame actual (contenido en "df_limpio") para verificar que hay solamente una columna más (por la división de 'Contract'), las demás se reemplazaron por las versiones mejoradas. Como se eliminaron las unidades de medida de las columnas en donde había datos de los jugadores (como por ejemplo el peso o el precio de mercado), se decidió incluirlas en el nombre de las columnas para seguir manteniendo la información. Se le echará un vistazo a las primeras 10 filas de todas las columnas modificadas para mostrar el resultado de la limpieza. Además, se cambiará el nombre de las columnas para que sean en minúscula y con separación por guion bajo ('_') en lugar de espacio. 

In [44]:
df_limpio.columns = df_limpio.columns.str.strip().str.lower().str.replace(' ', '_')

print("Dimensiones del DataFrame limpio:")
print(df_limpio.shape)

print("Primeras 10 filas del DataFrame limpio:")
print(df_limpio[['value_eur', 'wage_eur', 'release_clause_eur', 'height_cm', 'weight_kg', 'hits', 'contract_begins', 'joined', 'w/f', 'sm', 'ir']].head(10))

Dimensiones del DataFrame limpio:
(18979, 78)
Primeras 10 filas del DataFrame limpio:
     value_eur  wage_eur  release_clause_eur  height_cm  weight_kg    hits  \
0  103500000.0  560000.0         138400000.0      170.0       72.0   771.0   
1   63000000.0  220000.0          75900000.0      187.0       83.0   562.0   
2  120000000.0  125000.0         159400000.0      188.0       87.0   150.0   
3  129000000.0  370000.0         161000000.0      181.0       70.0   207.0   
4  132000000.0  270000.0         166500000.0      175.0       68.0   595.0   
5  111000000.0  240000.0         132000000.0      184.0       80.0   248.0   
6  120500000.0  250000.0         144300000.0      175.0       71.0   246.0   
7  102000000.0  160000.0         120300000.0      191.0       91.0   120.0   
8  185500000.0  160000.0         203100000.0      178.0       73.0  1600.0   
9  110000000.0  260000.0         147700000.0      187.0       85.0   130.0   

   contract_begins     joined  w/f  sm  ir  
0         